# Options and Result Stats

Use an analytical `to_*` function when you need clean data and stats. Every conversion returns a `Result`, and all analytical targets share the same input and cleaning options.

In [ ]:
import json
from pathlib import Path

import schema_sanitizer as ss

out_dir = Path('files/02_options_and_stats/exercise_01_stats')
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / 'events.jsonl'
path.write_text('{"id": 1, "ts": "2024/01/02 03:04:05"}\n', encoding='utf-8')


In [ ]:
result = ss.to_pyarrow(
    path,
    input_format='jsonl',
    custom_timestamp_patterns=(r'(\d{4})/(\d{2})/(\d{2}) (\d{2}):(\d{2}):(\d{2})',),
)
summary = {
    'rows': result.clean_data.num_rows,
    'schema': str(result.clean_data.schema),
    'stats': result.stats,
}
(out_dir / 'stats_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary


In [ ]:
run_1 = ss.to_pyarrow(
    path,
    input_format='jsonl',
    custom_timestamp_patterns=(r'(\d{4})/(\d{2})/(\d{2}) (\d{2}):(\d{2}):(\d{2})',),
)
run_2 = ss.to_pyarrow(
    path,
    input_format='jsonl',
    custom_timestamp_patterns=(r'(\d{4})/(\d{2})/(\d{2}) (\d{2}):(\d{2}):(\d{2})',),
)
# Per-run ETL metadata contains timestamps; compare deterministic business data and stats.
business_columns = ['id', 'ts']
assert run_1.clean_data.select(business_columns).equals(
    run_2.clean_data.select(business_columns)
)
assert run_1.stats == run_2.stats
